# 说明
这个示例需要搭配Aure的云服务，这里没有去改写和运行，大家了解即可

Azure AI Agent Service是一个完整的智能体托管和编排平台，project_client.agents.create_agent() 并不是在本地内存中创建一个 Agent，而是在 Azure AI Project 中创建一个持久化的 Agent 资源。代码的实际执行是在 Azure AI Agent Service 提供的安全、隔离的沙箱环境中完成的，而不是在您本地的机器上。

运行下面代码流程：
1. Azure 平台准备： 创建 Azure AI 项目（Project） -> 部署所需模型 -> 获取项目端点。
2. 本地环境准备： 安装 Python 依赖 -> 配置认证 -> 配置环境变量（.env 文件）。
3. 代码执行： 运行代码。



In [1]:
# 导入必要的库
import os  # 用于操作系统交互（如读取环境变量）
from dotenv import load_dotenv  # 用于加载 .env 文件中的环境变量

# Azure AI 相关库
from azure.ai.projects import AIProjectClient  # Azure AI 项目客户端
from azure.ai.agents.models import CodeInterpreterTool  # 代码解释器工具（用于执行代码生成图表等）
from azure.identity import DefaultAzureCredential  # Azure 默认身份认证（自动获取登录凭证）

# 类型提示和路径处理
from typing import Any  # 通用类型提示
from pathlib import Path  # 处理文件路径的跨平台工具
from datetime import datetime  # 处理日期时间（虽然本代码未直接使用）


In [ ]:
# 加载 .env 文件中的环境变量（通常存储敏感信息如 API 密钥）
load_dotenv()

# 从环境变量中获取 Azure 项目端点地址
project_endpoint = os.environ["PROJECT_ENDPOINT"]

# 创建 Azure AI 项目客户端
# 参数说明：
#   endpoint: Azure 项目的服务端点 URL
#   credential: 使用 DefaultAzureCredential 自动获取当前登录用户的 Azure 凭证
# 注意：这个客户端是后续所有操作的基础
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)

In [ ]:
from IPython.display import display, HTML, Image
from pathlib import Path


# 定义异步函数（使用 async/await 处理需要等待的操作，如网络请求）
async def run_agent_with_visualization():
    """
    执行 Azure AI Agent 并可视化结果的主函数
    核心目标：
        1. 实现一个能 自动生成数据图表 的 AI 助手。
        2. 演示 Azure AI Agent 框架 的完整工作流程（创建Agent -> 创建Thread -> 运行）。
        3. 展示 工具调用（Code Interpreter） 能力（Agent 调用 Python 代码生成图表）。

    内部流程示意：
    用户输入 → 创建对话线程 → Agent 接收请求
        ↓
    检测到需要生成图表 → 调用 Code Interpreter
        ↓
    生成并执行 Python 代码 → 保存图表文件（在 Azure AI Service 沙箱中）
        ↓
    将文件作为消息返回 → 前端渲染结果
    """

    # ----------------------------------------------------
    # 1. 初始化和Agent创建
    # ----------------------------------------------------

    # 初始化 HTML 输出字符串（用于在 Jupyter 中生成美观的可视化报告）
    html_output = "<h2>Azure AI Agent Execution</h2>"

    # 使用上下文管理器确保 project_client 资源正确释放（例如关闭网络连接）
    with project_client:
        # Create an instance of the CodeInterpreterTool
        # 创建代码解释器工具实例。它代表了一个可供 Agent 调用的、在安全沙箱中执行代码的能力
        code_interpreter = CodeInterpreterTool()

        # The CodeInterpreterTool needs to be included in creation of the agent
        # Ensure to set the correct model name as deployed in Azure AI Foundry for your use case
        # 创建 AI Agent（智能体）
        # Agent 是一个可持久化的实体，定义了其行为、模型和可用工具
        agent = project_client.agents.create_agent(
            model="gpt-4o",
            # model: 使用的 AI 模型（这里用 gpt-4o），必须是在 Azure AI Project 中已部署的模型名称
            name="my-agent",
            # name: Agent 的自定义名称
            instructions="You are helpful agent",
            # instructions: Agent 的行为指导（系统提示词），指导其如何响应请求
            tools=code_interpreter.definitions,
            # tools: 绑定的工具列表，将 CodeInterpreterTool 的定义（Schema）传递给 Agent
        )

        html_output += f"<div><strong>Created agent</strong> with ID: {agent.id}</div>"

        # ----------------------------------------------------
        # 2. 创建会话线程和发送用户消息
        # ----------------------------------------------------

        # Create a thread
        # 创建会话线程（Thread 相当于一次对话的上下文容器，保持状态）
        thread = project_client.agents.threads.create()
        html_output += f"<div><strong>Created thread</strong> with ID: {thread.id}</div>"

        # User query - display nicely
        # 用户查询内容（要求生成旅行者数据的柱状图）
        user_query = "Could you please create a bar chart for the operating profit using the following data and provide the file to me? Bali: 100 Travelers, Paris: 356 Travelers, London: 900 Travelers, Tokyo: 850 Travellers"
        # 将用户查询格式化为 HTML 显示（带样式）
        html_output += "<div style='margin:15px 0; padding:10px; background-color:#f5f5f5; border-left:4px solid #007bff; border-radius:4px;'>"
        html_output += "<strong>User:</strong><br>"
        html_output += f"<div style='margin-left:15px'>{user_query}</div>"
        html_output += "</div>"

        # Create a message
        # 在会话线程中创建用户消息，将查询内容添加到线程历史中
        message = project_client.agents.messages.create(
            thread_id=thread.id,
            role="user",
            content=user_query,
        )

        # ----------------------------------------------------
        # 3. 运行 Agent 并处理结果
        # ----------------------------------------------------

        # Run the agent - show a "processing" message
        # 先显示当前已有的 HTML 内容，并追加一个“处理中”提示
        display(HTML(
            html_output + "<div style='color:#007bff'><i>Processing request...</i></div>"))

        # Execute the run
        # 核心步骤：运行 Agent 来处理线程中的最新消息
        # create_and_process 会等待 Agent 执行完毕（包括可能的工具调用），直到状态变为 completed 或 failed
        run = project_client.agents.runs.create_and_process(
            thread_id=thread.id, agent_id=agent.id)

        # Update status
        # 记录运行状态
        status_color = 'green' if run.status == 'completed' else 'red'
        html_output += f"<div><strong>Run finished</strong> with status: <span style='color:{status_color}'>{run.status}</span></div>"

        if run.status == "failed":
            html_output += f"<div style='color:red'><strong>Run failed:</strong> {run.last_error}</div>"

        # Get messages from the thread
        # 获取线程中最新的所有消息（包括 Agent 的响应和文件信息）
        messages = project_client.agents.messages.list(thread_id=thread.id)

        # ----------------------------------------------------
        # 4. 解析和显示 Agent 响应
        # ----------------------------------------------------

        # Format assistant response
        html_output += "<div style='margin:15px 0; padding:10px; background-color:#f0f7ff; border-left:4px solid #28a745; border-radius:4px;'>"
        html_output += "<strong>Assistant:</strong><br>"

        # Handle messages based on the actual structure
        # First, try to get the assistant's text responses
        # --- 文本内容解析 ---
        # 尝试从返回的 messages 对象中提取 Agent (assistant) 的文本响应
        try:
            # First approach - if messages is a list of objects with role attribute
            # 尝试方法一：将 messages 视为一个列表，筛选出 role="assistant" 的消息
            assistant_msgs = [msg for msg in messages if hasattr(
                msg, 'role') and msg.role == "assistant"]

            if assistant_msgs:
                last_msg = assistant_msgs[-1] # 取最后一条 Agent 消息
                if hasattr(last_msg, 'content'):
                    if isinstance(last_msg.content, list):
                        # 如果 content 是一个列表（包含 text, image_file 等类型）
                        for content_item in last_msg.content:
                            if hasattr(content_item, 'type') and content_item.type == "text":
                                # 提取文本内容
                                html_output += f"<div style='margin-left:15px; white-space:pre-wrap'>{content_item.text.value}</div>"
                    elif isinstance(last_msg.content, str):
                        # 如果 content 是一个简单字符串
                        html_output += f"<div style='margin-left:15px; white-space:pre-wrap'>{last_msg.content}</div>"

            # 尝试方法二：如果方法一失败，尝试从 data 属性中查找（适配不同 SDK 版本的结构）
            if not assistant_msgs:
                if hasattr(messages, 'data'):
                    for msg in messages.data:
                        if hasattr(msg, 'role') and msg.role == "assistant":
                            if hasattr(msg, 'content'):
                                html_output += f"<div style='margin-left:15px; white-space:pre-wrap'>{msg.content}</div>"

        except Exception as e:
            html_output += f"<div style='color:red'><strong>Error processing messages:</strong> {str(e)}</div>"

        html_output += "</div>"

        # --- 图片附件解析 ---
        # Handle image contents based on the actual structure
        saved_images = []
        try:
            # 尝试访问 image_contents 属性，处理 Agent 返回的图片文件
            # Try to access image_contents as an attribute
            if hasattr(messages, 'image_contents'):
                for image_content in messages.image_contents:
                    file_id = image_content.image_file.file_id
                    file_name = f"{file_id}_image_file.png"
                    # 使用客户端方法将文件从 Azure 服务端下载到本地文件系统
                    project_client.agents.save_file(
                        file_id=file_id, file_name=file_name)
                    saved_images.append(file_name)
                    html_output += f"<div style='margin-top:10px'><strong>Generated Image:</strong> {file_name}</div>"
        except Exception as e:
            html_output += f"<div style='color:orange'><i>Note: No images found or error processing images</i></div>"

        # --- 文件路径注解解析 ---
        # Agent 在生成文件后，有时会在文本中返回一个占位符或路径注解
        # Handle file path annotations based on the actual structure
        try:
            # Try to access file_path_annotations as an attribute
            # 尝试访问 file_path_annotations 属性，处理文件路径注解
            if hasattr(messages, 'file_path_annotations'):
                for file_path_annotation in messages.file_path_annotations:
                    # 提取文件名
                    file_name = Path(file_path_annotation.text).name
                    
                    # 根据文件ID下载文件
                    project_client.agents.save_file(
                        file_id=file_path_annotation.file_path.file_id, file_name=file_name)
                    
                    # 格式化显示下载的文件信息
                    html_output += "<div style='margin:10px 0; padding:8px; background-color:#f8f9fa; border:1px solid #ddd; border-radius:4px;'>"
                    html_output += f"<strong>Generated File:</strong> {file_name}<br>"
                    html_output += f"<strong>Type:</strong> {file_path_annotation.type}<br>"
                    html_output += "</div>"
        except Exception as e:
            html_output += f"<div style='color:orange'><i>Note: No file annotations found or error processing files</i></div>"

        # ----------------------------------------------------
        # 5. 清理和显示最终结果
        # ----------------------------------------------------

        # Delete the agent once done
        # 清理资源：删除创建的 Agent（线程通常也会在一段时间后自动清理）
        project_client.agents.delete_agent(agent.id)
        html_output += "<div style='margin-top:10px'><i>Agent deleted after completion</i></div>"

        # Final display of all content
        # 最终显示所有 HTML 格式的输出结果
        display(HTML(html_output))

        # Display any saved images
        # 循环显示所有下载到本地的图片文件
        for img_file in saved_images:
            display(Image(img_file))

# Execute the function
# 在异步环境中（如 Jupyter Notebook 或 IPython）执行异步函数
await run_agent_with_visualization()

AttributeError: 'AgentsClient' object has no attribute 'list_messages'


---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
